<a href="https://colab.research.google.com/github/priyaakridhaa/fdp_day1_ml/blob/day1/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# We'll set this up properly in Part 2. This is just to build intuition.
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")   # tiny, free, ~80MB

a = model.encode("I love my puppy")
b = model.encode("My dog is adorable")
c = model.encode("Please file your taxes")

print("puppy vs dog :", round(float(util.cos_sim(a, b)), 2))   # ~0.6  (similar!)
print("puppy vs tax :", round(float(util.cos_sim(a, c)), 2))   # ~0.05 (different)

In [ ]:
!pip install -q groq chromadb sentence-transformers

In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
print("✅ Key loaded" if GROQ_API_KEY else "❌ Missing key")

In [ ]:
documents = [
    "Acme Corp offers 24 days of paid annual leave to all full-time employees.",
    "Employees can work from home up to 3 days per week with manager approval.",
    "The office is located at 42 MG Road, Bangalore, and opens at 9:00 AM.",
    "Acme Corp reimburses internet bills up to 1000 rupees per month for remote staff.",
    "New employees are on probation for the first 6 months of employment.",
    "The annual company retreat happens every December in Goa.",
]

In [ ]:
import chromadb

# In-memory database (resets when the notebook restarts — perfect for a lab)
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="acme_docs")

# Add our chunks. ChromaDB embeds them automatically. ✨
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],   # each chunk needs a unique id
)

print(f"✅ Stored {collection.count()} chunks in the vector database.")

In [ ]:
question = "what is the probation period of new employee?"   # note: doc says "annual leave", not "holidays"

results = collection.query(query_texts=[question], n_results=2)

print("🔎 Top matching chunks:")
for chunk in results["documents"][0]:
    print(" -", chunk)

In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"

def rag_answer(question: str, n_results: int = 2) -> str:
    # ① RETRIEVE the most relevant chunks
    results = collection.query(query_texts=[question], n_results=n_results)
    context = "\n".join(results["documents"][0])

    # ② AUGMENT: build a prompt that includes the context
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    # ③ GENERATE with Groq
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,   # keep it factual, not creative
    )
    return response.choices[0].message.content

# 🧪 Try it
print(rag_answer("How many holidays do I get?"))

In [ ]:
print("📖 Acme Doc Bot ready! Ask about leave, WFH, office, etc. Type 'quit' to exit.\n")

while True:
    q = input("You: ")
    if q.strip().lower() in {"quit", "exit", "q"}:
        print("👋 Bye!")
        break
    if not q.strip():
        continue
    print("Bot:", rag_answer(q), "\n")

📖 Acme Doc Bot ready! Ask about leave, WFH, office, etc. Type 'quit' to exit.

You: what is the probation period
Bot: The probation period is 6 months. 

